In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

In [2]:
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

print("Train Shape :", train_df.shape)
print("Test Shape  :", test_df.shape)

Train Shape : (690088, 15)
Test Shape  : (295753, 14)


In [3]:
X = train_df.drop(columns=["health_condition", "id"])
y = train_df["health_condition"]

test_ids = test_df["id"]
test_df = test_df.drop(columns=["id"])

print("X Shape :", X.shape)
print("y Shape :", y.shape)
print("Test Shape :", test_df.shape)

X Shape : (690088, 13)
y Shape : (690088,)
Test Shape : (295753, 13)


In [4]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

In [5]:
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):

    print("=" * 60)
    print(f"Fold {fold}")
    print("=" * 60)

    X_train = X.iloc[train_idx].copy()
    X_valid = X.iloc[valid_idx].copy()

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


In [6]:
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):

    print("=" * 60)
    print(f"Fold {fold}")
    print("=" * 60)

    # Split data
    X_train = X.iloc[train_idx].copy()
    X_valid = X.iloc[valid_idx].copy()

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    # -----------------------------
    # Missing Value Handling
    # -----------------------------

    # Numerical columns
    num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

    for col in num_cols:
        median = X_train[col].median()

        X_train[col] = X_train[col].fillna(median)
        X_valid[col] = X_valid[col].fillna(median)
        test_df[col] = test_df[col].fillna(median)

    # Categorical columns
    cat_cols = X_train.select_dtypes(include="object").columns.tolist()

    for col in cat_cols:
        mode = X_train[col].mode()[0]

        X_train[col] = X_train[col].fillna(mode)
        X_valid[col] = X_valid[col].fillna(mode)
        test_df[col] = test_df[col].fillna(mode)

    # -----------------------------
    # Class Weights
    # -----------------------------
    classes = np.unique(y_train)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    class_weights = dict(zip(classes, weights))

    # -----------------------------
    # CatBoost Model
    # -----------------------------
    model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    class_weights=class_weights,
    random_seed=42,

    task_type="GPU",
    devices="0",

    verbose=100
)

    # -----------------------------
    # Train
    # -----------------------------
    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        early_stopping_rounds=300
    )

    # -----------------------------msin
    # Validation
    # -----------------------------
    y_pred = model.predict(X_valid)

    score = balanced_accuracy_score(y_valid, y_pred)

    scores.append(score)

    print(f"Fold {fold}: {score:.5f}")

Fold 1
0:	learn: 0.7989964	test: 0.7990332	best: 0.7990332 (0)	total: 76.9ms	remaining: 6m 24s
100:	learn: 0.8959325	test: 0.8976451	best: 0.8976451 (100)	total: 3.08s	remaining: 2m 29s
200:	learn: 0.9066013	test: 0.9067604	best: 0.9067604 (200)	total: 6.07s	remaining: 2m 25s
300:	learn: 0.9090985	test: 0.9071770	best: 0.9072187 (272)	total: 9.01s	remaining: 2m 20s
400:	learn: 0.9110186	test: 0.9076547	best: 0.9079927 (386)	total: 11.9s	remaining: 2m 16s
500:	learn: 0.9125686	test: 0.9078473	best: 0.9081543 (475)	total: 14.9s	remaining: 2m 14s
600:	learn: 0.9139028	test: 0.9080033	best: 0.9082867 (540)	total: 17.9s	remaining: 2m 10s
700:	learn: 0.9149670	test: 0.9082513	best: 0.9082867 (540)	total: 20.7s	remaining: 2m 7s
800:	learn: 0.9162531	test: 0.9082654	best: 0.9084049 (707)	total: 23.7s	remaining: 2m 4s
900:	learn: 0.9175312	test: 0.9085215	best: 0.9086214 (893)	total: 26.7s	remaining: 2m 1s
1000:	learn: 0.9188567	test: 0.9084159	best: 0.9086455 (965)	total: 29.8s	remaining: 1m 5

In [16]:
print("=" * 60)
print("Cross Validation Results")
print("=" * 60)

for i, score in enumerate(scores, 1):
    print(f"Fold {i}: {score:.5f}")

print("=" * 60)
print(f"Mean Balanced Accuracy : {np.mean(scores):.5f}")
print(f"Standard Deviation     : {np.std(scores):.5f}")

Cross Validation Results
Fold 1: 0.90840
Fold 2: 0.90895
Fold 3: 0.90893
Fold 4: 0.90785
Fold 5: 0.90813
Mean Balanced Accuracy : 0.90845
Standard Deviation     : 0.00043


In [17]:
import os

os.makedirs("../models", exist_ok=True)

model.save_model(
    f"../output/models/catboost_fold_{fold}.cbm"
)

In [18]:
test_pred = model.predict_proba(test_df)

In [19]:
test_predictions = np.zeros((len(test_df), 3))

In [20]:
test_predictions += test_pred / skf.n_splits

In [21]:
classes = model.classes_

final_predictions = classes[np.argmax(test_predictions, axis=1)]

In [22]:
submission = pd.DataFrame({
    "id": test_ids,
    "health_condition": final_predictions
})

submission.to_csv(
    "../output/submissions/submission_cv_catboost.csv",
    index=False
)

print("Submission saved!")

Submission saved!
